<a href="https://colab.research.google.com/github/LINWOO0099/Categorical-Encoding/blob/main/task1_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import requests
import time
import json
import os
from datetime import datetime

# -------------------------------
# Configuration
# -------------------------------
BASE_URL = "https://hacker-news.firebaseio.com/v0"
HEADERS = {"User-Agent": "TrendPulse/1.0"}

# Category keywords
CATEGORIES = {
    "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
    "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
    "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
    "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
    "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
}

MAX_PER_CATEGORY = 25


# -------------------------------
# Helper: Categorize title
# -------------------------------
def get_category(title):
    title_lower = title.lower()

    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            if keyword in title_lower:
                return category
    return None


# -------------------------------
# Step 1: Fetch Top Story IDs
# -------------------------------
try:
    response = requests.get(f"{BASE_URL}/topstories.json", headers=HEADERS)
    response.raise_for_status()
    story_ids = response.json()[:500]
except Exception as e:
    print("Failed to fetch top stories:", e)
    exit()


# -------------------------------
# Step 2: Fetch Story Details
# -------------------------------
collected_data = []
category_count = {cat: 0 for cat in CATEGORIES}

for category in CATEGORIES:
    print(f"\nProcessing category: {category}")

    for story_id in story_ids:

        # Stop if category already has 25 stories
        if category_count[category] >= MAX_PER_CATEGORY:
            break

        try:
            story_res = requests.get(
                f"{BASE_URL}/item/{story_id}.json",
                headers=HEADERS
            )
            story_res.raise_for_status()
            story = story_res.json()

            # Skip if invalid
            if not story or "title" not in story:
                continue

            # Categorize
            assigned_category = get_category(story["title"])

            if assigned_category == category:

                data = {
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": assigned_category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                collected_data.append(data)
                category_count[category] += 1

        except Exception as e:
            print(f"Error fetching story {story_id}: {e}")
            continue

    # Wait 2 seconds after each category loop
    time.sleep(2)


# -------------------------------
# Step 3: Save JSON File
# -------------------------------
# Create folder if not exists
os.makedirs("data", exist_ok=True)

# Filename with date
today = datetime.now().strftime("%Y%m%d")
filename = f"data/trends_{today}.json"

try:
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(collected_data, f, indent=4)

    print(f"\nCollected {len(collected_data)} stories.")
    print(f"Saved to {filename}")

except Exception as e:
    print("Error saving file:", e)


Processing category: technology

Processing category: worldnews

Processing category: sports

Processing category: science

Processing category: entertainment

Collected 76 stories.
Saved to data/trends_20260404.json


In [11]:
import pandas as pd
import os

# Load JSON file
df = pd.read_json("data/trends_20260404.json")

# Clean data
df = df.drop_duplicates(subset=["post_id"])
df = df.fillna(0)

# Save as CSV
os.makedirs("data", exist_ok=True)
df.to_csv("data/trends_cleaned.csv", index=False)

print(" Done! CSV file created.")

 Done! CSV file created.
